# Selective facts Stage 2 — baseline1
Raw dataset + request → fresh contract where applicable → generation → verification/repair → evaluation.
No previous parses, pilot outputs, Stage 1 thresholds, cached responses, GitHub push or other notebook outputs are needed.
Select **GPU T4 ×2**, Internet on. Only proposed/ablation need the new screened **sf1 1.5B adapter** attached as a private dataset.
This version uses the sf1 selective-facts input protocol. Train the matching adapter with selective_facts_train.ipynb first and inspect its screening result. Live parsers receive only supported request-relevant facts. Conventional checks use hand-set thresholds. The original 100 test tasks are diagnostic after prior failure analysis; these are not fresh confirmatory results.
Run a two-task infrastructure smoke check, then all 100 tasks in this notebook. Use a fresh Kaggle session.

In [ ]:
import base64, hashlib, json, os, subprocess, sys, zlib
from pathlib import Path
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
PAYLOAD_SHA='2bcf40613e6cd6830c326a5c05ee7d9a7a3eebd4fd046f3df1bac64f9a448c5c'
PAYLOAD=(
'...'
'g7pl/dv//f8Ab0Jx9g==')
raw=zlib.decompress(base64.b64decode(PAYLOAD))
assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA
ROOT=WORK/('stage2_selective_source_'+PAYLOAD_SHA[:12])
for name,encoded_source in json.loads(raw).items():
    source=base64.b64decode(encoded_source)
    relative=Path(name)
    assert not relative.is_absolute() and '..' not in relative.parts
    path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():assert path.read_bytes()==source, f'Changed embedded source: {path}'
    else:path.write_bytes(source)
PACKAGES=WORK/'stage2_selective_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')
print('Source and raw inputs:',PAYLOAD_SHA)


In [ ]:
PRESET='baseline1'
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
# PEFT probes Kaggle's optional torchao package even for this ordinary LoRA adapter.
# Check adapter injection on a tiny CPU model before downloading the large models.
subprocess.run([sys.executable,'-m','eval.stage2_selective.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
ADAPTER=None
if PRESET in ['proposed','no_contract_to_generator']:
    matches=[]
    for marker in Path('/kaggle/input').rglob('input_protocol.json'):
        try:
            m=json.loads(marker.read_text());selection=json.loads((marker.parent/'selection.json').read_text())
            if m.get('protocol')=='p1-v2+sf1' and selection.get('eligible') is True and selection.get('smoke') is False:matches.append(marker.parent)
        except (ValueError,OSError):pass
    assert len(matches)==1, f'Attach exactly one sf1 adapter that passed full screening; found {matches}. Original or smoke adapters cannot be used.'
    ADAPTER=matches[0]
    subprocess.run([sys.executable,'-c','import sys; from finetune.selective_facts.adapter import validate_adapter; print(validate_adapter(sys.argv[1]))',str(ADAPTER)],cwd=ROOT,env=ENV,check=True)
print('System:',PRESET,'Adapter:',ADAPTER)


In [ ]:
def execute(smoke):
    name='stage2_selective_'+PRESET+('_smoke' if smoke else '')
    out=WORK/name
    command=[sys.executable,'-m','eval.stage2_selective.runner','--preset',PRESET,'--out',str(out),'--packages',str(PACKAGES)]
    if ADAPTER is not None:command+=['--adapter',str(ADAPTER)]
    if smoke:command+=['--limit','2']
    log=WORK/(name+'.log')
    with log.open('w') as f:
        process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');f.write(line);f.flush()
        code=process.wait()
    assert code==0, f'Run failed; download {log} and the partial ZIP. A partial job is not a result.'
    import zipfile
    with zipfile.ZipFile(out.with_suffix('.zip')) as z:assert json.loads(z.read('bundle.json'))['complete']
    assert len(list((out/'tasks').glob('*.json')))==(2 if smoke else 100)
    return out.with_suffix('.zip')
execute(smoke=True)


In [ ]:
BUNDLE=execute(smoke=False)
from IPython.display import FileLink,display
display(FileLink(str(BUNDLE)))
print('COMPLETE AND VALIDATED: 100 / 100')


Download the full ZIP and executed notebook. Keep them in `kaggle_outputs/stage2/selective_facts_v1/` and `notebooks/executed/stage2/selective_facts_v1/`. Run each of the six notebooks independently; then attach their six full ZIPs to `stage2_selective_collect.ipynb` (CPU).